[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/34.%20The%20Hazard%20%26%20IMDG%20Segregation%20Problem/P34-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 34. The Hazard & IMDG Segregation Problem

## Tier 3: Particle Swarm Optimization

### Goal and scope

This notebook applies Particle Swarm Optimization (PSO) to a small container-stowage instance. The decision is where to place each container so that incompatible dangerous-goods classes are separated while the placement also favors spacing and a balanced distribution across holds.

The input consists of containers and their IMDG classes, candidate stowage positions, and pairwise segregation requirements. The output is a feasible-or-best-found assignment, its fitness score, violation count, and diagnostic plots. PSO is stochastic: it searches for a good solution but does not prove that the returned solution is globally optimal or even feasible.

### Modeling assumptions

- Each container is assigned to exactly one position, and each position can hold at most one container. Unused positions are allowed.
- Position coordinates are measured in metres. The five positions within each hold are spaced 3 m apart along the horizontal axis; hold rows are 6 m apart along the vertical axis. Euclidean distance between these coordinates is used for distance-based segregation checks.
- A requirement marked `different_hold` is enforced as a separate hard condition in addition to any minimum-distance condition.
- The fitness function also includes a general spacing preference and hold-balance preference. These preferences are objectives, not additional IMDG rules.
- The example uses a simplified set of segregation requirements. It is an educational model, not a substitute for applicable regulations, vessel-specific constraints, or operational review.

### How PSO represents this problem

A particle stores one position index for each container. PSO updates these indices using real-valued velocities; rounding and a repair step convert the updated vector into a discrete assignment with no repeated positions. Personal best means the best assignment found so far by one particle; global best means the best assignment found so far by the entire swarm.

For container indices \(i,j\), let \(x_i\) be the position index assigned to container \(i\), and let \(d_{x_i x_j}\) be the Euclidean distance between their positions. The assignment must satisfy

$$
x_i \in \{0,\ldots,m-1\}, \qquad x_i \ne x_j \quad (i \ne j),
$$

where \(m\) is the number of candidate positions. The score minimized by the swarm is

$$
F(x) = w_1 P_{\mathrm{viol}}(x)
+ w_2 P_{\mathrm{dist}}(x)
+ w_3 P_{\mathrm{hold}}(x).
$$

Here, \(P_{\mathrm{viol}}\) sums severity penalties for violated segregation requirements; \(P_{\mathrm{dist}}\) sums \(\max(0, 6-d_{x_i x_j})\) over container pairs; and \(P_{\mathrm{hold}}\) sums each hold's absolute deviation from the ideal average number of containers per hold. The weights \(w_1,w_2,w_3\) control the relative importance of these terms. A lower score is preferred, but the score is not a probability and does not certify feasibility by itself. The code reports the number of segregation violations separately.

### PSO update and discrete repair

At each iteration, particle \(p\)'s velocity and position are updated componentwise:

$$
v_p^{(t+1)} = \omega v_p^{(t)}
+ c_1 r_1 \bigl(b_p-x_p^{(t)}\bigr)
+ c_2 r_2 \bigl(g-x_p^{(t)}\bigr),
\qquad
x_p^{(t+1)} = x_p^{(t)} + v_p^{(t+1)}.
$$

The inertia weight \(\omega\) carries motion forward; \(c_1\) and \(c_2\) weight attraction toward the particle's personal best \(b_p\) and the swarm's global best \(g\); and \(r_1,r_2\) are independently sampled random values in \([0,1]\). Because assignments are discrete and must use distinct positions, each update is bounded, rounded, and repaired to remove duplicate position indices.

For \(n\) containers and swarm size \(S\), evaluating every particle examines \(O(n^2)\) container pairs, so the fitness-evaluation work is \(O(Sn^2)\) per iteration. The repair step also maintains a unique position for every container. The number of iterations and swarm size determine the main computational effort; PSO does not enumerate or certify all possible assignments.

### What to examine

- The best-so-far fitness history shows whether the swarm found better assignments during the run.
- The segregation-violation count indicates whether the returned assignment satisfies the modeled pairwise rules.
- Swarm diversity describes spread in the particles' position-index vectors; it is a search diagnostic, not a measure of solution quality.
- The sensitivity experiment illustrates how changing PSO coefficients can affect outcomes on this stochastic instance. Individual runs need not reproduce one another because no random seed is specified.

In [ ]:
from typing import Dict, List, Tuple
from dataclasses import dataclass
import random
import time

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class Container:
    """A container carrying dangerous goods."""

    id: str
    imdg_class: str
    weight: float = 20.0


@dataclass
class Position:
    """A stowage position with coordinates in metres."""

    id: str
    hold: int
    x: float
    y: float
    z: float


@dataclass
class SegregationRequirement:
    """A pairwise segregation rule for two IMDG classes."""

    class1: str
    class2: str
    requirement: str
    min_distance: float
    different_hold: bool = False


@dataclass
class Particle:
    """A PSO candidate with current and personal-best assignments."""

    position: np.ndarray
    velocity: np.ndarray
    fitness: float
    best_position: np.ndarray
    best_fitness: float


@dataclass
class PSOConfig:
    """Parameters controlling the particle swarm search."""

    swarm_size: int = 30
    max_iterations: int = 50
    inertia_weight: float = 0.7
    cognitive_coeff: float = 1.5
    social_coeff: float = 1.5
    max_velocity: float = 4.0

In [ ]:
def create_pso_example():
    """Create the 12-container, 20-position PSO demonstration instance."""

    imdg_classes = [
        "1.1", "1.4S", "3", "3", "8", "8.1",
        "2.1", "2.3", "5.1", "6.1", "9", "9",
    ]
    containers = [
        Container(f"C{i + 1}", imdg_class, 20.0)
        for i, imdg_class in enumerate(imdg_classes)
    ]

    # Global coordinates keep positions in different holds spatially distinct.
    positions = []
    for hold in range(1, 5):
        for pos_idx in range(5):
            x = pos_idx * 3.0
            y = (hold - 1) * 6.0
            positions.append(
                Position(f"H{hold}_P{pos_idx + 1}", hold, x, y, 0.0)
            )

    segregation_requirements = [
        SegregationRequirement("1.1", "3", "Away From", 6.0),
        SegregationRequirement("1.1", "8", "Separated From", 0.0, True),
        SegregationRequirement("1.1", "2.1", "Away From", 6.0),
        SegregationRequirement("1.1", "2.3", "Away From", 6.0),
        SegregationRequirement("1.1", "5.1", "Separated From", 0.0, True),
        SegregationRequirement("1.1", "6.1", "Separated From", 0.0, True),
        SegregationRequirement("1.1", "9", "Away From", 3.0),
        SegregationRequirement("1.4S", "3", "Away From", 3.0),
        SegregationRequirement("1.4S", "8", "Separated From", 0.0, True),
        SegregationRequirement("1.4S", "2.1", "Away From", 3.0),
        SegregationRequirement("1.4S", "2.3", "Away From", 3.0),
        SegregationRequirement("3", "8", "Away From", 3.0),
        SegregationRequirement("3", "2.1", "No restriction", 0.0),
        SegregationRequirement("3", "2.3", "Away From", 3.0),
        SegregationRequirement("3", "5.1", "Away From", 3.0),
        SegregationRequirement("3", "6.1", "Away From", 3.0),
        SegregationRequirement("3", "9", "No restriction", 0.0),
        SegregationRequirement("8", "2.1", "Away From", 3.0),
        SegregationRequirement("8", "2.3", "Away From", 3.0),
        SegregationRequirement("8", "5.1", "Away From", 3.0),
        SegregationRequirement("8", "6.1", "Away From", 3.0),
        SegregationRequirement("8", "9", "Away From", 3.0),
        SegregationRequirement("2.1", "2.3", "Away From", 3.0),
        SegregationRequirement("2.1", "5.1", "Away From", 3.0),
        SegregationRequirement("2.1", "6.1", "Away From", 3.0),
        SegregationRequirement("2.1", "9", "No restriction", 0.0),
        SegregationRequirement("5.1", "6.1", "Away From", 3.0),
        SegregationRequirement("5.1", "9", "Away From", 3.0),
        SegregationRequirement("6.1", "9", "Away From", 3.0),
    ]

    return containers, positions, segregation_requirements


containers, positions, seg_reqs = create_pso_example()

print("PSO problem instance")
print(f"  Containers: {len(containers)}")
print(f"  Positions: {len(positions)}")
print(f"  Segregation requirements: {len(seg_reqs)}")

print("\nContainer classes")
class_counts = {}
for container in containers:
    class_counts[container.imdg_class] = (
        class_counts.get(container.imdg_class, 0) + 1
    )
for imdg_class, count in sorted(class_counts.items()):
    print(f"  Class {imdg_class}: {count} containers")

print("\nHold distribution")
hold_counts = {}
for position in positions:
    hold_counts[position.hold] = hold_counts.get(position.hold, 0) + 1
for hold, count in sorted(hold_counts.items()):
    print(f"  Hold {hold}: {count} positions")

In [ ]:
def calculate_distance_matrix(
    positions: List[Position],
) -> Dict[Tuple[str, str], float]:
    """Compute Euclidean distances for every ordered pair of positions."""

    distances = {}
    for i, pos1 in enumerate(positions):
        for j, pos2 in enumerate(positions):
            if i != j:
                delta_x = pos1.x - pos2.x
                delta_y = pos1.y - pos2.y
                delta_z = pos1.z - pos2.z
                distances[(pos1.id, pos2.id)] = float(
                    np.sqrt(delta_x**2 + delta_y**2 + delta_z**2)
                )
    return distances


def find_applicable_requirement(
    container1: Container,
    container2: Container,
    requirements: List[SegregationRequirement],
):
    """Return the rule for a class pair, independent of pair order."""

    for requirement in requirements:
        direct_match = (
            requirement.class1 == container1.imdg_class
            and requirement.class2 == container2.imdg_class
        )
        reverse_match = (
            requirement.class1 == container2.imdg_class
            and requirement.class2 == container1.imdg_class
        )
        if direct_match or reverse_match:
            return requirement
    return None


def check_segregation_compliance(
    container1: Container,
    container2: Container,
    pos1: Position,
    pos2: Position,
    requirements: List[SegregationRequirement],
    distances: Dict[Tuple[str, str], float],
) -> bool:
    """Check the applicable distance and different-hold conditions."""

    requirement = find_applicable_requirement(
        container1, container2, requirements
    )
    if requirement is None or requirement.requirement == "No restriction":
        return True

    distance = distances[(pos1.id, pos2.id)]
    if distance < requirement.min_distance:
        return False
    if requirement.different_hold and pos1.hold == pos2.hold:
        return False
    return True


distance_matrix = calculate_distance_matrix(positions)

In [ ]:
def calculate_fitness(
    position_vector: np.ndarray,
    containers: List[Container],
    positions: List[Position],
    requirements: List[SegregationRequirement],
    distances: Dict[Tuple[str, str], float],
    w1: float = 1000.0,
    w2: float = 1.0,
    w3: float = 10.0,
) -> float:
    """Return the weighted score for a complete, unique assignment."""

    position_vector = np.asarray(position_vector)
    if position_vector.shape != (len(containers),):
        raise ValueError("The assignment must contain one index per container.")

    rounded_indices = np.rint(position_vector).astype(int)
    if not np.allclose(position_vector, rounded_indices):
        raise ValueError("Assignment indices must be integers.")
    if np.any(rounded_indices < 0) or np.any(rounded_indices >= len(positions)):
        raise ValueError("An assignment index is outside the position range.")
    if len(set(rounded_indices)) != len(containers):
        raise ValueError("Each container must have a distinct position.")

    assignments = {
        container.id: positions[position_idx]
        for container, position_idx in zip(containers, rounded_indices)
    }

    violation_penalty = 0.0
    for i, container1 in enumerate(containers):
        for j in range(i + 1, len(containers)):
            container2 = containers[j]
            pos1 = assignments[container1.id]
            pos2 = assignments[container2.id]
            if check_segregation_compliance(
                container1, container2, pos1, pos2, requirements, distances
            ):
                continue

            requirement = find_applicable_requirement(
                container1, container2, requirements
            )
            if requirement is not None and requirement.different_hold and (
                pos1.hold == pos2.hold
            ):
                violation_penalty += 100.0
            elif requirement is not None and requirement.min_distance > 0:
                distance = distances[(pos1.id, pos2.id)]
                violation_penalty += max(
                    0.0, requirement.min_distance - distance
                ) * 10.0

    total_distance_penalty = 0.0
    for i, container1 in enumerate(containers):
        for j in range(i + 1, len(containers)):
            container2 = containers[j]
            pos1 = assignments[container1.id]
            pos2 = assignments[container2.id]
            distance = distances[(pos1.id, pos2.id)]
            if distance < 6.0:
                total_distance_penalty += 6.0 - distance

    hold_ids = sorted({position.hold for position in positions})
    ideal_per_hold = len(containers) / len(hold_ids)
    hold_usage = {hold: 0 for hold in hold_ids}
    for assignment in assignments.values():
        hold_usage[assignment.hold] += 1
    hold_imbalance = sum(
        abs(hold_usage[hold] - ideal_per_hold) for hold in hold_ids
    )

    return (
        w1 * violation_penalty
        + w2 * total_distance_penalty
        + w3 * hold_imbalance
    )


def repair_position_vector(
    position_vector: np.ndarray,
    num_positions: int,
) -> np.ndarray:
    """Bound and round indices, then reassign duplicates to unused positions."""

    repaired = np.rint(position_vector).astype(int)
    repaired = np.clip(repaired, 0, num_positions - 1)
    unique_positions, counts = np.unique(repaired, return_counts=True)
    duplicate_positions = unique_positions[counts > 1]

    for duplicate in duplicate_positions:
        duplicate_indices = np.where(repaired == duplicate)[0]
        for index in duplicate_indices[1:]:
            available_positions = sorted(
                set(range(num_positions)) - set(repaired)
            )
            if not available_positions:
                raise ValueError("There are not enough positions for all containers.")
            repaired[index] = random.choice(available_positions)

    return repaired.astype(float)


def initialize_particle(
    swarm_size: int,
    num_containers: int,
    num_positions: int,
) -> List[Particle]:
    """Create particles with unique initial assignments and small velocities."""

    if num_containers > num_positions:
        raise ValueError("There must be at least one position per container.")

    particles = []
    for _ in range(swarm_size):
        position = np.random.choice(
            num_positions, size=num_containers, replace=False
        ).astype(float)
        velocity = np.random.uniform(-1.0, 1.0, num_containers)
        particles.append(
            Particle(
                position=position,
                velocity=velocity,
                fitness=float("inf"),
                best_position=position.copy(),
                best_fitness=float("inf"),
            )
        )
    return particles

In [ ]:
def particle_swarm_optimization(
    containers: List[Container],
    positions: List[Position],
    requirements: List[SegregationRequirement],
    distances: Dict[Tuple[str, str], float],
    config: PSOConfig,
) -> Dict:
    """Search for a low-fitness assignment using particle swarm updates."""

    if config.swarm_size < 1 or config.max_iterations < 1:
        raise ValueError("Swarm size and iteration count must be positive.")
    if config.max_velocity < 0:
        raise ValueError("Maximum velocity must be non-negative.")

    start_time = time.time()
    num_containers = len(containers)
    num_positions = len(positions)
    particles = initialize_particle(
        config.swarm_size, num_containers, num_positions
    )

    global_best_position = None
    global_best_fitness = float("inf")
    convergence_history = []
    diversity_history = []

    print(
        f"PSO algorithm: swarm size {config.swarm_size}, "
        f"maximum iterations {config.max_iterations}"
    )
    print("=" * 70)

    for iteration in range(config.max_iterations):
        for particle in particles:
            particle.fitness = calculate_fitness(
                particle.position,
                containers,
                positions,
                requirements,
                distances,
            )
            if particle.fitness < particle.best_fitness:
                particle.best_fitness = particle.fitness
                particle.best_position = particle.position.copy()

        for particle in particles:
            if particle.fitness < global_best_fitness:
                global_best_fitness = particle.fitness
                global_best_position = particle.position.copy()

        convergence_history.append(global_best_fitness)
        positions_matrix = np.array([particle.position for particle in particles])
        diversity_history.append(float(np.mean(np.std(positions_matrix, axis=0))))

        for particle in particles:
            r1 = np.random.random(num_containers)
            r2 = np.random.random(num_containers)
            cognitive_component = (
                config.cognitive_coeff
                * r1
                * (particle.best_position - particle.position)
            )
            social_component = (
                config.social_coeff
                * r2
                * (global_best_position - particle.position)
            )
            particle.velocity = (
                config.inertia_weight * particle.velocity
                + cognitive_component
                + social_component
            )
            particle.velocity = np.clip(
                particle.velocity,
                -config.max_velocity,
                config.max_velocity,
            )
            candidate_position = particle.position + particle.velocity
            particle.position = repair_position_vector(
                candidate_position, num_positions
            )

        if (iteration + 1) % 10 == 0 or iteration == 0:
            print(
                f"Iteration {iteration + 1:3d}: "
                f"best fitness = {global_best_fitness:.2f}, "
                f"diversity = {diversity_history[-1]:.2f}"
            )

    # Evaluate the final swarm positions so the reported distribution is current.
    for particle in particles:
        particle.fitness = calculate_fitness(
            particle.position,
            containers,
            positions,
            requirements,
            distances,
        )

    final_assignments = {
        container.id: positions[int(position_idx)]
        for container, position_idx in zip(containers, global_best_position)
    }

    final_violations = 0
    for i, container1 in enumerate(containers):
        for j in range(i + 1, len(containers)):
            container2 = containers[j]
            pos1 = final_assignments[container1.id]
            pos2 = final_assignments[container2.id]
            if not check_segregation_compliance(
                container1, container2, pos1, pos2, requirements, distances
            ):
                final_violations += 1

    return {
        "best_position": global_best_position,
        "best_fitness": global_best_fitness,
        "assignments": final_assignments,
        "violations": final_violations,
        "convergence_history": convergence_history,
        "diversity_history": diversity_history,
        "execution_time": time.time() - start_time,
        "particles": particles,
    }

In [ ]:
pso_config = PSOConfig(
    swarm_size=30,
    max_iterations=50,
    inertia_weight=0.7,
    cognitive_coeff=1.5,
    social_coeff=1.5,
    max_velocity=4.0,
)

print("IMDG segregation: particle swarm optimization")
print("=" * 60)
print(f"Problem: {len(containers)} containers, {len(positions)} positions")
print(
    f"PSO configuration: {pso_config.swarm_size} particles, "
    f"{pso_config.max_iterations} iterations"
)

pso_result = particle_swarm_optimization(
    containers, positions, seg_reqs, distance_matrix, pso_config
)

random_assignment = np.random.choice(
    len(positions), size=len(containers), replace=False
).astype(float)
random_fitness = calculate_fitness(
    random_assignment, containers, positions, seg_reqs, distance_matrix
)
if random_fitness > 0:
    improvement = (
        (random_fitness - pso_result["best_fitness"]) / random_fitness * 100.0
    )
else:
    improvement = 0.0

print("\n" + "=" * 60)
print("PSO results")
print("=" * 60)
print(f"Best fitness: {pso_result['best_fitness']:.2f}")
print(f"Segregation violations: {pso_result['violations']}")
print(f"Execution time: {pso_result['execution_time']:.4f} seconds")
print(
    f"Containers placed: {len(pso_result['assignments'])}/{len(containers)}"
)
print("\nSingle random-assignment comparison")
print(f"Random-assignment fitness: {random_fitness:.2f}")
print(f"PSO best fitness: {pso_result['best_fitness']:.2f}")
print(f"Relative score reduction: {improvement:.1f}%")

In [ ]:
def visualize_pso_results(
    pso_result: Dict,
    containers: List[Container],
    positions: List[Position],
    requirements: List[SegregationRequirement],
    distances: Dict[Tuple[str, str], float],
) -> None:
    """Plot search diagnostics, stowage, and pairwise compliance details."""

    iterations = range(1, len(pso_result["convergence_history"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(
        iterations, pso_result["convergence_history"], color="navy", linewidth=2
    )
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel("Best-so-far fitness (score)")
    axes[0].set_title("Best-so-far fitness")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(
        iterations, pso_result["diversity_history"], color="green", linewidth=2
    )
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel("Mean standard deviation of position indices")
    axes[1].set_title("Swarm diversity")
    axes[1].grid(True, alpha=0.3)

    final_fitnesses = [particle.fitness for particle in pso_result["particles"]]
    axes[2].hist(
        final_fitnesses,
        bins=15,
        color="skyblue",
        edgecolor="black",
        alpha=0.8,
    )
    axes[2].axvline(
        pso_result["best_fitness"],
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Best-so-far: {pso_result['best_fitness']:.2f}",
    )
    axes[2].set_xlabel("Fitness score")
    axes[2].set_ylabel("Number of particles")
    axes[2].set_title("Final swarm fitness distribution")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

    class_colors = {
        "1.1": "darkred",
        "1.4S": "red",
        "3": "orange",
        "8": "purple",
        "8.1": "darkviolet",
        "2.1": "green",
        "2.3": "darkgreen",
        "5.1": "brown",
        "6.1": "olive",
        "9": "blue",
    }
    containers_by_id = {container.id: container for container in containers}
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    for hold, ax in zip(sorted({pos.hold for pos in positions}), axes.flat):
        ax.set_title(f"Hold {hold}: final placement")
        ax.set_xlabel("Horizontal position (m)")
        ax.set_ylabel("Local vertical position (m)")
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-1, 13)
        ax.set_ylim(-1, 3)
        hold_positions = [pos for pos in positions if pos.hold == hold]

        for pos in hold_positions:
            assigned_container = next(
                (
                    containers_by_id[container_id]
                    for container_id, assigned_pos in pso_result["assignments"].items()
                    if assigned_pos.id == pos.id
                ),
                None,
            )
            local_y = pos.y - (pos.hold - 1) * 6.0
            if assigned_container is None:
                ax.scatter(
                    pos.x, local_y, s=300, c="lightgray", marker="s",
                    edgecolors="black", label="Empty position",
                )
                ax.text(pos.x, local_y, "Empty", ha="center", va="center", fontsize=6)
            else:
                color = class_colors.get(assigned_container.imdg_class, "gray")
                ax.scatter(
                    pos.x, local_y, s=300, c=color, marker="s",
                    edgecolors="black",
                )
                ax.text(
                    pos.x,
                    local_y,
                    f"{assigned_container.id}\n{assigned_container.imdg_class}",
                    ha="center",
                    va="center",
                    fontsize=7,
                )

    present_classes = sorted({container.imdg_class for container in containers})
    legend_handles = [
        plt.Line2D(
            [0], [0], marker="s", linestyle="", color="w",
            markerfacecolor=class_colors.get(imdg_class, "gray"),
            markeredgecolor="black", label=f"IMDG class {imdg_class}",
        )
        for imdg_class in present_classes
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=5)
    fig.suptitle("Final container placement by hold")
    fig.tight_layout(rect=(0, 0.1, 1, 0.95))
    plt.show()

    violation_details = []
    for i, container1 in enumerate(containers):
        for j in range(i + 1, len(containers)):
            container2 = containers[j]
            pos1 = pso_result["assignments"][container1.id]
            pos2 = pso_result["assignments"][container2.id]
            requirement = find_applicable_requirement(
                container1, container2, requirements
            )
            compliant = check_segregation_compliance(
                container1, container2, pos1, pos2, requirements, distances
            )
            violation_details.append(
                {
                    "container1": container1,
                    "container2": container2,
                    "pos1": pos1,
                    "pos2": pos2,
                    "requirement": (
                        requirement.requirement
                        if requirement is not None
                        else "No listed rule"
                    ),
                    "min_distance": (
                        requirement.min_distance
                        if requirement is not None
                        else 0.0
                    ),
                    "distance": distances[(pos1.id, pos2.id)],
                    "compliant": compliant,
                }
            )

    violation_details.sort(
        key=lambda detail: (
            detail["compliant"],
            detail["requirement"] == "No listed rule",
            detail["container1"].id,
            detail["container2"].id,
        )
    )
    shown_details = violation_details[:20]
    table_data = [[
        "Container pair", "Classes", "Requirement", "Positions",
        "Minimum distance (m)", "Actual distance (m)", "Status",
    ]]
    cell_colors = [["#4472C4"] * 7]
    for detail in shown_details:
        table_data.append(
            [
                f"{detail['container1'].id} - {detail['container2'].id}",
                f"{detail['container1'].imdg_class} - {detail['container2'].imdg_class}",
                detail["requirement"],
                f"{detail['pos1'].id} - {detail['pos2'].id}",
                f"{detail['min_distance']:.1f}",
                f"{detail['distance']:.1f}",
                "Compliant" if detail["compliant"] else "Violation",
            ]
        )
        status_color = "lightgreen" if detail["compliant"] else "lightcoral"
        cell_colors.append(["white"] * 6 + [status_color])

    fig, ax = plt.subplots(figsize=(14, max(4, 0.35 * len(table_data))))
    ax.axis("off")
    table = ax.table(
        cellText=table_data,
        cellColours=cell_colors,
        cellLoc="center",
        loc="center",
        colWidths=[0.12, 0.1, 0.15, 0.2, 0.13, 0.13, 0.12],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.5)
    for column in range(7):
        table[(0, column)].set_text_props(weight="bold", color="white")
    ax.set_title(
        f"Pairwise compliance details (showing {len(shown_details)} of "
        f"{len(violation_details)} pairs)",
        pad=15,
    )
    plt.show()


visualize_pso_results(
    pso_result, containers, positions, seg_reqs, distance_matrix
)

In [ ]:
def parameter_sensitivity_analysis():
    """Compare selected PSO coefficient settings on the same instance."""

    parameter_sets = [
        {
            "inertia_weight": 0.9,
            "cognitive_coeff": 2.0,
            "social_coeff": 2.0,
            "name": "High exploration",
        },
        {
            "inertia_weight": 0.7,
            "cognitive_coeff": 1.5,
            "social_coeff": 1.5,
            "name": "Balanced",
        },
        {
            "inertia_weight": 0.5,
            "cognitive_coeff": 1.0,
            "social_coeff": 2.5,
            "name": "Social focus",
        },
        {
            "inertia_weight": 0.4,
            "cognitive_coeff": 2.5,
            "social_coeff": 1.0,
            "name": "Cognitive focus",
        },
    ]
    results = []

    print("Parameter sensitivity analysis")
    print("=" * 50)
    for parameters in parameter_sets:
        print(f"\nTesting: {parameters['name']}")
        test_config = PSOConfig(
            swarm_size=20,
            max_iterations=30,
            inertia_weight=parameters["inertia_weight"],
            cognitive_coeff=parameters["cognitive_coeff"],
            social_coeff=parameters["social_coeff"],
            max_velocity=4.0,
        )
        test_result = particle_swarm_optimization(
            containers, positions, seg_reqs, distance_matrix, test_config
        )

        history = test_result["convergence_history"]
        if len(history) >= 10 and not np.isclose(history[0], 0.0):
            convergence_rate = (history[0] - history[9]) / history[0] * 100.0
        else:
            convergence_rate = 0.0

        results.append(
            {
                "name": parameters["name"],
                "final_fitness": test_result["best_fitness"],
                "violations": test_result["violations"],
                "convergence_rate": convergence_rate,
                "final_diversity": test_result["diversity_history"][-1],
            }
        )
        print(f"  Final fitness: {test_result['best_fitness']:.2f}")
        print(f"  Segregation violations: {test_result['violations']}")
        print(f"  First-ten-iteration score reduction: {convergence_rate:.1f}%")

    names = [result["name"] for result in results]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    fitnesses = [result["final_fitness"] for result in results]
    bars = axes[0, 0].bar(names, fitnesses, color="skyblue", alpha=0.8)
    axes[0, 0].set_ylabel("Final fitness (score)")
    axes[0, 0].set_title("Best-found score")
    axes[0, 0].grid(True, alpha=0.3, axis="y")
    for bar, value in zip(bars, fitnesses):
        axes[0, 0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.1f}",
            ha="center",
            va="bottom",
        )

    violations = [result["violations"] for result in results]
    violation_colors = [
        "green" if value == 0 else "orange" if value <= 2 else "red"
        for value in violations
    ]
    bars = axes[0, 1].bar(names, violations, color=violation_colors, alpha=0.8)
    axes[0, 1].set_ylabel("Number of segregation violations")
    axes[0, 1].set_title("Feasibility diagnostic")
    axes[0, 1].grid(True, alpha=0.3, axis="y")
    for bar, value in zip(bars, violations):
        axes[0, 1].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(value),
            ha="center",
            va="bottom",
        )

    convergence_rates = [result["convergence_rate"] for result in results]
    axes[1, 0].bar(names, convergence_rates, color="lightgreen", alpha=0.8)
    axes[1, 0].set_ylabel("Score reduction over first 10 iterations (%)")
    axes[1, 0].set_title("Early best-so-far improvement")
    axes[1, 0].grid(True, alpha=0.3, axis="y")

    diversities = [result["final_diversity"] for result in results]
    axes[1, 1].bar(names, diversities, color="orange", alpha=0.8)
    axes[1, 1].set_ylabel("Mean standard deviation of position indices")
    axes[1, 1].set_title("Final swarm diversity")
    axes[1, 1].grid(True, alpha=0.3, axis="y")

    for ax in axes.flat:
        plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
    fig.suptitle("PSO parameter sensitivity")
    fig.tight_layout()
    plt.show()

    print("\nSensitivity summary")
    print("=" * 50)
    for result in results:
        print(f"\n{result['name']}:")
        print(f"  Final fitness: {result['final_fitness']:.2f}")
        print(f"  Segregation violations: {result['violations']}")
        print(f"  First-ten-iteration score reduction: {result['convergence_rate']:.1f}%")
        print(f"  Final swarm diversity: {result['final_diversity']:.2f}")


parameter_sensitivity_analysis()

## Interpreting the method

PSO searches with a population of candidate assignments. Each particle uses its own search history and the swarm's best-known assignment to guide subsequent updates. This population-based process can explore several parts of the assignment space, but its stochastic updates do not provide an optimality guarantee. Different runs can return different assignments and scores.

The fitness history is a best-so-far record: it can stay constant when no better assignment is found, but it should not increase. A zero segregation-violation count means that the returned assignment satisfies the pairwise rules represented in this instance. A positive count means the assignment is not feasible under those rules, regardless of how its score compares with another assignment. Distance and hold-balance terms distinguish assignments with the same violation penalty; they do not replace the hard interpretation of the reported violation count.

The sensitivity experiment changes the inertia, cognitive, and social coefficients while retaining the same problem data and search structure. Since the runs are stochastic and use no fixed seed, differences between parameter settings reflect both parameter effects and random variation; a single run per setting is illustrative rather than a statistical performance comparison.

### Strengths and limitations

**Potential strengths**
- The swarm keeps multiple candidate assignments in play and combines individual and shared search information.
- The same fitness-evaluation and repair steps can be applied as the instance grows, subject to increased computational effort.
- The method can be useful when a good assignment is sought without exhaustive enumeration.

**Important limitations**
- PSO is stochastic and does not certify global optimality or guarantee a violation-free assignment.
- Results depend on the fitness weights, update coefficients, repair rule, and random draws.
- The simplified geometry and segregation data omit operational details such as vessel-specific access, compatibility exceptions, and regulatory interpretation.
- A model-based assignment requires validation against the applicable rules and operational constraints before practical use.

### When this approach is useful

This tier is useful for studying population-based optimization on a discrete placement problem and for exploring trade-offs between segregation violations, spacing, and hold balance. Additional real-world constraints or larger instances can be represented within the same modeling framework, but each extension requires its own carefully validated data and assumptions.